## Simple Baseline: synthetic FinPay payments

This cell builds a fake dataset of **2,000 P2P payments** with a fixed random seed (42), so the results are the same on every run.

- **`amount`**: payment size drawn from an exponential distribution (mean ≈ 50). Most payments are small and a few are large.
- **`is_new_recipient`**: 1 if the payer has never paid this person before, otherwise 0 (about a 50/50 split).
- **`is_fraud`**: the label. A payment is fraud only when `amount > 150` **and** `is_new_recipient == 1`.

The last line prints the share of each class: about **97% legitimate and 3% fraud** (97.25% / 2.75% with this seed). 

In [6]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)
n = 2000
df = pd.DataFrame({
    "amount": rng.exponential(scale=50, size=n).round(2),
    "is_new_recipient": rng.integers(0, 2, size=n),
})
df["is_fraud"] = ((df["amount"] > 150) & (df["is_new_recipient"] == 1)).astype(int)
print(df["is_fraud"].value_counts(normalize=True))

is_fraud
0    0.9725
1    0.0275
Name: proportion, dtype: float64


### Predict, then run: rule-based baseline

This cell uses the simplest possible "model": a one-line rule that flags a payment as fraud when `amount > 150` **and** it goes to a new recipient. It then scores the rule with:

- **Precision**: of the payments we flagged, how many were really fraud?
- **Recall**: of all the real fraud, how much did we catch?

The rule is the same formula that created the labels, so expect both scores to be a perfect **1.0**. Real fraud is never this clean.

In [7]:
df["rule_pred"] = ((df["amount"] > 150) & (df["is_new_recipient"] == 1)).astype(int)

from sklearn.metrics import precision_score, recall_score
print("Precision:", precision_score(df["is_fraud"], df["rule_pred"]))
print("Recall:   ", recall_score(df["is_fraud"], df["rule_pred"]))

Precision: 1.0
Recall:    1.0


### 1.2: One real model, one line of comparison

This cell trains a **logistic regression** model to learn the fraud pattern from the data instead of using a hand-written rule.

- `train_test_split` holds back **30%** of the rows as a test set, so the model is scored on payments it never saw during training (`random_state=1` keeps the split the same on every run).
- `.fit()` learns from the training rows, and `.predict()` labels the test rows as fraud (1) or not (0).
- Precision and recall are then computed on the test set. `zero_division=0` returns 0 instead of a warning if the model flags nothing at all.

Logistic regression draws a single straight line through the features, so it can't exactly copy an "amount > 150 **AND** new recipient" rule. Expect scores below the rule's perfect 1.0, then decide: **ship the rule or ship the model, and why?**

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

X_train, X_test, y_train, y_test = train_test_split(
    df[["amount", "is_new_recipient"]], df["is_fraud"], test_size=0.3, random_state=1
)
model = LogisticRegression().fit(X_train, y_train)
preds = model.predict(X_test)
print("Model precision:", precision_score(y_test, preds, zero_division=0))
print("Model recall:   ", recall_score(y_test, preds, zero_division=0))

Model precision: 0.8
Model recall:    0.6666666666666666


### Verdict: ship the rule (for now)

| | Precision | Recall |
|---|---|---|
| Rule | 1.00 | 1.00 |
| Logistic regression | 0.80 | 0.67 |

**Ship the rule.** On this data it is perfect, cheap to run, and easy to explain. The model raises false alarms (1 in 5 flags is wrong) and misses about a third of the fraud, because a single straight-line boundary can't copy an "amount > 150 **AND** new recipient" rule exactly.

**The catch:** the rule only wins because it is the same formula that created the labels. On real data, nobody knows the true rule, and fraud patterns change. There, a model that learns from data, outputs a risk score (so we can tune the threshold or rank payments for review), and is retrained on fresh data would be the better choice. Neither a rule nor a trained model adapts on its own, so both need monitoring for drift.

## Drift Catch: simulating a change in the data

In production, incoming data can quietly change shape over time. This is called **data drift**. This cell fakes it:

- It copies the two input features into a new table, `drifted`, leaving the original `df` untouched.
- It multiplies every `amount` by **1.5**, as if average spending crept up by 50% over a few months.
- It leaves `is_new_recipient` unchanged.

The prints compare the average `amount` before and after. Expect the drifted average to be exactly 1.5× the original.

**Why it matters:** the rule and the model both use a fixed cut-off around `amount > 150`. With bigger amounts, many more normal payments cross that line, so the number of fraud alerts jumps even though no extra fraud is happening. Nothing errors or crashes, which is why drift needs a monitor to catch it.

In [11]:
drifted = df[["amount", "is_new_recipient"]].copy()
drifted["amount"] = drifted["amount"] * 1.5   # a few months later, average spend has crept up
print("Original average amount:", df["amount"].mean())
print("Drifted average amount: ", drifted["amount"].mean())

Original average amount: 50.749415
Drifted average amount:  76.1241225


In [14]:
# Simple drift monitor: alert if a feature's average has grown by more than a set percentage.

def check_feature_drift(new_avg, baseline_avg, pct_threshold=0.2):
    # Return True if new_avg is more than pct_threshold (e.g. 20%) bigger than baseline_avg
    pct_change = (new_avg - baseline_avg) / baseline_avg
    return pct_change > pct_threshold

fires = check_feature_drift(drifted["amount"].mean(), df["amount"].mean())
print("Drift alert fired:", fires)

Drift alert fired: True


## Quick Format Check: CSV vs Parquet

**Guess first:** which file will be smaller?

This cell saves the same `df` table in two formats and prints each file's size in KB (bytes ÷ 1024):

- **CSV** (`data.csv`): plain text, one line per row, with values separated by commas. Anyone can open it in a text editor or Excel. `index=False` skips writing pandas' row numbers as an extra column.
- **Parquet** (`data.parquet`): a compressed binary format that stores data **column by column** and remembers each column's data type. It needs the `pyarrow` package installed earlier.

Parquet is usually much smaller, because values in the same column look alike and compress well (for example, `is_new_recipient` and `is_fraud` are only 0s and 1s). With a tiny table like this, Parquet's built-in metadata takes up some of the space, so the gap is smaller than it would be on millions of rows.

`data.csv` is also the input for the Mage AI pipeline later on.

**30-second discussion:** which format would you hand to a colleague to eyeball in a text editor? Which would you use to store years of transaction history?

In [15]:
import os
df.to_csv("data.csv", index=False)
df.to_parquet("data.parquet")

print("CSV size (KB):    ", os.path.getsize("data.csv") / 1024)
print("Parquet size (KB):", os.path.getsize("data.parquet") / 1024)

CSV size (KB):     25.1376953125
Parquet size (KB): 15.1328125
